# Beyond Goals & Assists: Measuring Rodri's True Impact — World Cup 2026

## Project Objective

This project analyzes Rodri's contribution during the 2026 World Cup beyond traditional attacking metrics such as goals and assists.

### Main Research Question

**How can we measure the true impact of a central midfielder beyond goals and assists?**

The analysis focuses on six dimensions:

1. Ball circulation
2. Line-breaking and progression
3. Defensive contribution
4. Pressing
5. Off-ball availability
6. Physical output

We also use **mplsoccer** to create football-specific spatial visualizations such as pass maps, touch maps, heatmaps, final-third entry maps, and shot maps.

> Important methodological note: this CSV is a merged file containing multiple sub-datasets. We must separate them using the `source_dataset` column before analyzing them.


# Phase 0 — Install and Import Libraries

We first install `mplsoccer` and import the libraries required for data manipulation and visualization.


In [ ]:
!pip install mplsoccer -q


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from mplsoccer import Pitch

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 200)


# Phase 1 — Load the Dataset

Upload the merged CSV to Colab if needed.

The notebook first looks for the expected file name in the current Colab session. If it is not found, it opens the Colab upload widget.


In [ ]:
from pathlib import Path

possible_files = [
    Path("/content/rodri.csv"),
    Path("rodri.csv"),
    Path("/mnt/data/rodri.csv"),
    Path("/content/rodri_wc2026_all_datasets_merged (1)(1).csv"),
    Path("/content/rodri_wc2026_all_datasets_merged (1).csv"),
    Path("/content/rodri_wc2026_all_datasets_merged.csv"),
]

file_path = None

for path in possible_files:
    if path.exists():
        file_path = path
        break

if file_path is None:
    from google.colab import files
    uploaded = files.upload()
    file_path = Path(next(iter(uploaded.keys())))

print("Using file:", file_path)


In [ ]:
df = pd.read_csv(file_path)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()


# Phase 2 — Understand the Merged Dataset

This file is not one homogeneous table. It contains several sub-datasets stacked vertically.

The column `source_dataset` tells us where each row originally came from.


In [ ]:
df.info()


In [ ]:
print("Duplicated rows:", df.duplicated().sum())

df["source_dataset"].value_counts()


## Why this matters

The same football action may appear in more than one sub-dataset.

For example, a pass may appear inside:

- the Opta event dataset,
- the pass dataset,
- the touch dataset.

Therefore, we should **never count all event rows from the merged dataframe without filtering by `source_dataset` first**.


# Phase 3 — Separate the Main Sub-Datasets


In [ ]:
match_df = df[
    df["source_dataset"] == "rodri_wc2026_match_by_match.csv"
].copy()

passes = df[
    df["source_dataset"] == "rodri_wc2026_passes.csv"
].copy()

touches = df[
    df["source_dataset"] == "rodri_wc2026_touches.csv"
].copy()

shots = df[
    df["source_dataset"] == "rodri_wc2026_shots.csv"
].copy()

events = df[
    df["source_dataset"] == "rodri_wc2026_events_opta.csv"
].copy()

line_breaks = df[
    df["source_dataset"] == "rodri_wc2026_line_breaks.csv"
].copy()

physical = df[
    df["source_dataset"] == "rodri_wc2026_physical.csv"
].copy()

defensive = df[
    df["source_dataset"] == "rodri_wc2026_out_of_possession.csv"
].copy()

offers = df[
    df["source_dataset"] == "rodri_wc2026_offers_receptions.csv"
].copy()


In [ ]:
print("Matches:", len(match_df))
print("Passes:", len(passes))
print("Touches:", len(touches))
print("Shots:", len(shots))
print("Events:", len(events))
print("Line-break rows:", len(line_breaks))
print("Physical rows:", len(physical))
print("Defensive rows:", len(defensive))
print("Offer/reception rows:", len(offers))


# Phase 4 — Data Cleaning

We clean percentage columns and standardize opponent names across the sub-datasets.


In [ ]:
percentage_columns = [
    "pass_completion_pct",
    "line_break_completion_pct"
]

for col in percentage_columns:
    if col in match_df.columns:
        match_df[col] = (
            match_df[col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .replace("nan", np.nan)
        )
        match_df[col] = pd.to_numeric(match_df[col], errors="coerce")


In [ ]:
opponent_mapping = {
    "Arabie Saoudite": "Saudi Arabia",
    "Autriche": "Austria",
    "Belgique": "Belgium",
    "Argentine": "Argentina"
}

for data in [passes, touches, shots, events]:
    if "opponent" in data.columns:
        data["opponent"] = data["opponent"].replace(opponent_mapping)


In [ ]:
# Convert important spatial and pass variables to numeric
numeric_cols = ["x", "y", "endX", "endY", "qual_Length", "qual_Angle"]

for data in [passes, touches, shots, events]:
    for col in numeric_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")


# Phase 5 — Tournament Overview

We inspect the eight matches contained in the match-level dataset.


In [ ]:
match_df[
    ["match_no", "opponent", "group"]
].reset_index(drop=True)


# Phase 6 — Tournament KPI Summary

This phase creates the core tournament statistics used throughout the project.


In [ ]:
total_passes = match_df["passes_attempted"].sum()
completed_passes = match_df["passes_completed"].sum()

pass_accuracy = (
    completed_passes / total_passes * 100
    if total_passes > 0 else np.nan
)

line_breaks_attempted = match_df["line_breaks_attempted"].sum()
line_breaks_completed = match_df["line_breaks_completed"].sum()

line_break_accuracy = (
    line_breaks_completed / line_breaks_attempted * 100
    if line_breaks_attempted > 0 else np.nan
)

total_distance = match_df["total_distance_m"].sum() / 1000

total_regains = match_df["possession_regains"].sum()

total_offers = match_df["total_offers"].sum()
offers_received = match_df["offers_received"].sum()

print("Matches:", len(match_df))
print("Passes attempted:", int(total_passes))
print("Passes completed:", int(completed_passes))
print("Pass accuracy:", round(pass_accuracy, 2), "%")
print()
print("Line breaks attempted:", int(line_breaks_attempted))
print("Line breaks completed:", int(line_breaks_completed))
print("Line-break accuracy:", round(line_break_accuracy, 2), "%")
print()
print("Possession regains:", int(total_regains))
print("Total distance:", round(total_distance, 2), "km")
print("Total offers:", int(total_offers))
print("Offers received:", int(offers_received))


In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Matches",
        "Passes Attempted",
        "Passes Completed",
        "Pass Accuracy (%)",
        "Line Breaks Attempted",
        "Line Breaks Completed",
        "Line Break Accuracy (%)",
        "Possession Regains",
        "Tackles Won",
        "Total Distance (km)",
        "Offers",
        "Offers Received",
        "Shots",
        "Goals"
    ],
    "Value": [
        len(match_df),
        int(match_df["passes_attempted"].sum()),
        int(match_df["passes_completed"].sum()),
        round(pass_accuracy, 2),
        int(match_df["line_breaks_attempted"].sum()),
        int(match_df["line_breaks_completed"].sum()),
        round(line_break_accuracy, 2),
        int(match_df["possession_regains"].sum()),
        int(match_df["tackles_won"].sum()),
        round(total_distance, 2),
        int(match_df["total_offers"].sum()),
        int(match_df["offers_received"].sum()),
        int(match_df["attempts_at_goal"].sum()),
        int(match_df["goals"].sum())
    ]
})

summary


# Phase 7 — Passing Analysis

Passing volume and pass completion help us understand Rodri's role in controlling possession.


In [ ]:
pass_stats = match_df.set_index("opponent")[
    ["passes_attempted", "passes_completed"]
]

pass_stats.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Rodri — Passing Volume by Match")
plt.xlabel("Opponent")
plt.ylabel("Number of Passes")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    match_df["opponent"],
    match_df["pass_completion_pct"],
    marker="o"
)

plt.axhline(
    match_df["pass_completion_pct"].mean(),
    linestyle="--",
    label="Tournament Average"
)

plt.title("Rodri — Pass Completion by Match")
plt.xlabel("Opponent")
plt.ylabel("Pass Completion (%)")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


# Phase 8 — Line-Breaking Analysis

Passing accuracy alone does not tell us whether a midfielder progresses the ball.

Line-breaking actions measure how often Rodri successfully moves the ball around, over, or through opposition lines.


In [ ]:
linebreak_stats = match_df.set_index("opponent")[
    ["line_breaks_attempted", "line_breaks_completed"]
]

linebreak_stats.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Rodri — Line-Breaking Actions by Match")
plt.xlabel("Opponent")
plt.ylabel("Line Breaks")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
linebreak_types = pd.Series({
    "Around": line_breaks["direction_around"].sum(),
    "Over": line_breaks["direction_over"].sum(),
    "Through": line_breaks["direction_through"].sum()
})

linebreak_types


In [ ]:
plt.figure(figsize=(8, 5))

linebreak_types.plot(kind="bar")

plt.title("Rodri — Types of Line-Breaking Actions")
plt.xlabel("Direction")
plt.ylabel("Number of Line Breaks")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# Phase 9 — Defensive Contribution

We combine possession regains, interceptions, tackles, clearances, and blocks to describe Rodri's defensive impact.


In [ ]:
defensive_stats = match_df.set_index("opponent")[
    ["possession_regains", "interceptions", "tackles_won"]
]

defensive_stats.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Rodri — Defensive Contribution by Match")
plt.xlabel("Opponent")
plt.ylabel("Actions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
defensive_totals = pd.Series({
    "Possession Regains": match_df["possession_regains"].sum(),
    "Interceptions": match_df["interceptions"].sum(),
    "Tackles Won": match_df["tackles_won"].sum(),
    "Clearances": match_df["clearances"].sum(),
    "Blocks": match_df["blocks"].sum()
})

defensive_totals


In [ ]:
plt.figure(figsize=(9, 5))

defensive_totals.plot(kind="bar")

plt.title("Rodri — Tournament Defensive Actions")
plt.ylabel("Total Actions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# Phase 10 — Pressing Analysis

Pressing actions help capture defensive influence without the ball.


In [ ]:
pressing = pd.Series({
    "Direct Pressing": match_df["pressing_direct"].sum(),
    "Indirect Pressing": match_df["pressing_indirect"].sum(),
    "Pushing On": match_df["pushing_on"].sum()
})

pressing


In [ ]:
plt.figure(figsize=(8, 5))

pressing.plot(kind="bar")

plt.title("Rodri — Pressing Contribution")
plt.ylabel("Actions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


# Phase 11 — Physical Performance

We analyze total distance covered and top speed by match.


In [ ]:
match_df["distance_km"] = match_df["total_distance_m"] / 1000

plt.figure(figsize=(12, 5))

plt.bar(
    match_df["opponent"],
    match_df["distance_km"]
)

plt.title("Rodri — Distance Covered by Match")
plt.xlabel("Opponent")
plt.ylabel("Distance (km)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Total distance:", round(match_df["distance_km"].sum(), 2), "km")
print("Average distance per match:", round(match_df["distance_km"].mean(), 2), "km")


In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    match_df["opponent"],
    match_df["top_speed_kmh"],
    marker="o"
)

plt.title("Rodri — Top Speed by Match")
plt.xlabel("Opponent")
plt.ylabel("Top Speed (km/h)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# Phase 12 — Off-Ball Availability

An elite midfielder contributes even before receiving the ball.

The dataset contains the number of times Rodri offers himself as a passing option and the number of times he actually receives the ball.


In [ ]:
offer_stats = match_df.set_index("opponent")[
    ["total_offers", "offers_received"]
]

offer_stats.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Rodri — Off-Ball Availability")
plt.xlabel("Opponent")
plt.ylabel("Number of Actions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
offer_rate = (
    match_df["offers_received"].sum()
    / match_df["total_offers"].sum()
    * 100
)

print("Offers received rate:", round(offer_rate, 2), "%")


# Phase 13 — Event-Level EDA

Now we use only the dedicated Opta event dataset.


In [ ]:
event_counts = events["event"].value_counts()

event_counts.head(15)


In [ ]:
plt.figure(figsize=(10, 6))

event_counts.head(10).sort_values().plot(
    kind="barh"
)

plt.title("Rodri — Most Frequent Event Types")
plt.xlabel("Number of Events")
plt.tight_layout()
plt.show()


# Phase 14 — Prepare Pass Data for mplsoccer


In [ ]:
successful = passes[
    passes["outcome"] == "Successful"
].copy()

unsuccessful = passes[
    passes["outcome"] == "Unsuccessful"
].copy()

print("Successful:", len(successful))
print("Unsuccessful:", len(unsuccessful))
print("Event-level pass accuracy:", round(len(successful) / len(passes) * 100, 2), "%")


# Phase 15 — mplsoccer: All Passes

The dataset uses Opta-style 0–100 pitch coordinates, so we use `pitch_type="opta"`.

Because there are many passes, this chart is intentionally dense. Match-level pass maps will be easier to interpret.


In [ ]:
pitch = Pitch(
    pitch_type="opta",
    pitch_color="white",
    line_color="black"
)

fig, ax = pitch.draw(figsize=(12, 8))

pitch.arrows(
    successful["x"],
    successful["y"],
    successful["endX"],
    successful["endY"],
    ax=ax,
    width=1,
    alpha=0.20,
    color="green",
    label="Successful"
)

pitch.arrows(
    unsuccessful["x"],
    unsuccessful["y"],
    unsuccessful["endX"],
    unsuccessful["endY"],
    ax=ax,
    width=1,
    alpha=0.35,
    color="red",
    label="Unsuccessful"
)

ax.set_title(
    "Rodri — All Passes | World Cup 2026",
    fontsize=16
)

ax.legend()
plt.show()


# Phase 16 — Reusable Match Pass Map

This function lets us generate a pass map for any opponent.


In [ ]:
def plot_pass_map(opponent):

    match_passes = passes[
        passes["opponent"] == opponent
    ].copy()

    successful_match = match_passes[
        match_passes["outcome"] == "Successful"
    ]

    unsuccessful_match = match_passes[
        match_passes["outcome"] == "Unsuccessful"
    ]

    pitch = Pitch(
        pitch_type="opta",
        pitch_color="white",
        line_color="black"
    )

    fig, ax = pitch.draw(figsize=(12, 8))

    pitch.arrows(
        successful_match["x"],
        successful_match["y"],
        successful_match["endX"],
        successful_match["endY"],
        ax=ax,
        color="green",
        alpha=0.5,
        width=1.2,
        label="Successful"
    )

    pitch.arrows(
        unsuccessful_match["x"],
        unsuccessful_match["y"],
        unsuccessful_match["endX"],
        unsuccessful_match["endY"],
        ax=ax,
        color="red",
        alpha=0.7,
        width=1.2,
        label="Unsuccessful"
    )

    ax.set_title(
        f"Rodri Pass Map vs {opponent}",
        fontsize=16
    )

    ax.legend()
    plt.show()


In [ ]:
plot_pass_map("Argentina")


# Phase 17 — Touch Map


In [ ]:
pitch = Pitch(
    pitch_type="opta",
    pitch_color="white",
    line_color="black"
)

fig, ax = pitch.draw(figsize=(12, 8))

pitch.scatter(
    touches["x"],
    touches["y"],
    ax=ax,
    alpha=0.25,
    s=20
)

ax.set_title(
    "Rodri — Touch Locations | World Cup 2026",
    fontsize=16
)

plt.show()


# Phase 18 — Touch Heatmap

A heatmap makes Rodri's main areas of involvement easier to identify than a large scatter plot.


In [ ]:
pitch = Pitch(
    pitch_type="opta",
    pitch_color="white",
    line_color="black"
)

fig, ax = pitch.draw(figsize=(12, 8))

bin_statistic = pitch.bin_statistic(
    touches["x"],
    touches["y"],
    statistic="count",
    bins=(6, 5)
)

pitch.heatmap(
    bin_statistic,
    ax=ax,
    cmap="Reds",
    edgecolors="white"
)

pitch.label_heatmap(
    bin_statistic,
    ax=ax,
    color="black",
    fontsize=10
)

ax.set_title(
    "Rodri — Touch Heatmap | World Cup 2026",
    fontsize=16
)

plt.show()


# Phase 18B — Eight Match-by-Match Touch Heatmaps (4 × 2 Football Pitches)

Visualize **Rodri's spatial involvement in each of his eight World Cup 2026 matches**. Each subplot represents one match (identified by `match_no`) and uses only rows from the **touches** dataset. Coordinates are in Opta's 0–100 system.

- The **same color scale** is used across all eight pitches, so color intensity is comparable between matches.
- Each heatmap counts observed touches in pitch zones, **not time spent in those zones**. This is not a GPS tracking heatmap.
- Touch counts are displayed in each title. Both the touch locations and match labels come directly from the CSV.


In [ ]:
# Eight individual match heatmaps: 2 rows × 4 football pitches.
# Uses the previously prepared `touches` and `match_df` DataFrames.
import matplotlib.pyplot as plt
import matplotlib as mpl
from mplsoccer import Pitch

# Identify matches from the match-level dataset, rather than relying on
# inconsistent opponent spelling across other sub-datasets.
matches_for_heatmaps = (
    match_df[["match_no", "opponent"]]
    .dropna(subset=["match_no"])
    .drop_duplicates(subset=["match_no"])
    .sort_values("match_no")
    .head(8)
)

if len(matches_for_heatmaps) != 8:
    raise ValueError(f"Expected 8 matches; found {len(matches_for_heatmaps)}")

# Clean positions without changing the source data.
touch_locations = touches.copy()
for col in ["match_no", "x", "y"]:
    touch_locations[col] = pd.to_numeric(touch_locations[col], errors="coerce")
touch_locations = touch_locations.dropna(subset=["match_no", "x", "y"])
touch_locations = touch_locations[
    touch_locations["x"].between(0, 100)
    & touch_locations["y"].between(0, 100)
]

pitch = Pitch(pitch_type="opta", pitch_color="#f9fbf9", line_color="#54635a", linewidth=1)
BINS = (12, 8)  # Shared zone layout for all matches

# Compute all heatmaps first, to establish a single global color scale.
heatmap_data = []
for match in matches_for_heatmaps.itertuples(index=False):
    match_touches = touch_locations[touch_locations["match_no"] == match.match_no]
    stats = pitch.bin_statistic(
        match_touches["x"].to_numpy(),
        match_touches["y"].to_numpy(),
        statistic="count", bins=BINS,
    )
    heatmap_data.append((int(match.match_no), str(match.opponent), len(match_touches), stats))

max_count = max(float(np.nanmax(item[3]["statistic"])) for item in heatmap_data)
max_count = max(max_count, 1.0)

fig, axs = pitch.draw(nrows=2, ncols=4, figsize=(22, 11))
fig.patch.set_facecolor("white")

for ax, (match_no, opponent, n_touches, stats) in zip(axs.flat, heatmap_data):
    pitch.heatmap(
        stats, ax=ax, cmap="YlOrRd", vmin=0, vmax=max_count,
        edgecolors="#ffffff", linewidth=0.25, alpha=0.9,
    )
    ax.set_title(
        f"Match {match_no} · vs {opponent}\n{n_touches} touches",
        fontsize=12, fontweight="semibold", pad=10,
    )

# One shared legend communicates the same number of touches per zone.
norm = mpl.colors.Normalize(vmin=0, vmax=max_count)
sm = mpl.cm.ScalarMappable(norm=norm, cmap="YlOrRd")
sm.set_array([])
fig.subplots_adjust(left=0.025, right=0.985, bottom=0.12, top=0.88, wspace=0.12, hspace=0.28)
cbar_ax = fig.add_axes([0.30, 0.07, 0.40, 0.018])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Rodri touches per pitch zone (shared scale)", fontsize=11)
fig.suptitle("Rodri | World Cup 2026 — Touch Heatmap by Match", fontsize=19, fontweight="bold", y=0.97)
plt.show()


# Phase 19 — Successful Final-Third Entries

We create a simple derived metric.

### Definition used in this notebook

A pass is considered a successful final-third entry when:

- it starts before the final third (`x < 66.7`);
- it ends inside the final third (`endX >= 66.7`);
- the pass is successful.

This is a transparent project definition, not an official Opta metric.


In [ ]:
final_third_entries = passes[
    (passes["x"] < 66.7)
    & (passes["endX"] >= 66.7)
    & (passes["outcome"] == "Successful")
].copy()

print("Successful final-third entries:", len(final_third_entries))


In [ ]:
pitch = Pitch(
    pitch_type="opta",
    pitch_color="white",
    line_color="black"
)

fig, ax = pitch.draw(figsize=(12, 8))

pitch.arrows(
    final_third_entries["x"],
    final_third_entries["y"],
    final_third_entries["endX"],
    final_third_entries["endY"],
    ax=ax,
    width=1.3,
    alpha=0.6
)

ax.axvline(
    66.7,
    linestyle="--"
)

ax.set_title(
    "Rodri — Successful Final-Third Entries",
    fontsize=16
)

plt.show()


# Phase 20 — Shot Analysis


In [ ]:
available_shot_cols = [
    col for col in ["opponent", "minute", "x", "y", "event", "outcome"]
    if col in shots.columns
]

shots[available_shot_cols]


In [ ]:
pitch = Pitch(
    pitch_type="opta",
    pitch_color="white",
    line_color="black"
)

fig, ax = pitch.draw(figsize=(12, 8))

pitch.scatter(
    shots["x"],
    shots["y"],
    ax=ax,
    s=150,
    edgecolor="black"
)

ax.set_title(
    "Rodri — Shot Map | World Cup 2026",
    fontsize=16
)

plt.show()


# Phase 21 — Passing Characteristics


In [ ]:
qualifier_cols = [
    "qual_KeyPass",
    "qual_Cross",
    "qual_Longball",
    "qual_Throughball",
    "qual_BigChanceCreated"
]

for col in qualifier_cols:
    if col in passes.columns:
        passes[col] = pd.to_numeric(passes[col], errors="coerce").fillna(0)

passing_types = pd.Series({
    "Key Passes": passes["qual_KeyPass"].sum(),
    "Crosses": passes["qual_Cross"].sum(),
    "Long Balls": passes["qual_Longball"].sum(),
    "Through Balls": passes["qual_Throughball"].sum(),
    "Big Chances Created": passes["qual_BigChanceCreated"].sum()
})

passing_types


In [ ]:
plt.figure(figsize=(9, 5))

passing_types.plot(kind="bar")

plt.title("Rodri — Passing Characteristics")
plt.ylabel("Number of Passes")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# Phase 22 — Pass Length Distribution


In [ ]:
passes["qual_Length"] = pd.to_numeric(
    passes["qual_Length"],
    errors="coerce"
)

plt.figure(figsize=(10, 5))

plt.hist(
    passes["qual_Length"].dropna(),
    bins=20,
    edgecolor="black"
)

plt.title("Rodri — Pass Length Distribution")
plt.xlabel("Pass Length")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


# Phase 23 — Match-by-Match Performance Summary


In [ ]:
match_summary = match_df[
    [
        "opponent",
        "passes_attempted",
        "passes_completed",
        "pass_completion_pct",
        "line_breaks_completed",
        "possession_regains",
        "tackles_won",
        "pressing_direct",
        "pressing_indirect",
        "total_offers",
        "offers_received",
        "distance_km",
        "top_speed_kmh"
    ]
].copy()

match_summary.round(2)


# Phase 24 — Consistency Analysis

Descriptive statistics help us see how stable Rodri's performances were across the eight matches.


In [ ]:
consistency_metrics = match_df[
    [
        "pass_completion_pct",
        "line_breaks_completed",
        "possession_regains",
        "tackles_won",
        "distance_km"
    ]
]

consistency_metrics.describe().round(2)


# Phase 25 — Correlation Analysis

Because we only have eight matches, correlations should be interpreted as exploratory rather than statistically conclusive.


In [ ]:
correlation_data = match_df[
    [
        "passes_attempted",
        "pass_completion_pct",
        "line_breaks_completed",
        "possession_regains",
        "pressing_direct",
        "total_offers",
        "distance_km"
    ]
]

correlation = correlation_data.corr()

correlation.round(2)


In [ ]:
plt.figure(figsize=(9, 7))

plt.imshow(
    correlation,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.colorbar()

plt.xticks(
    range(len(correlation.columns)),
    correlation.columns,
    rotation=90
)

plt.yticks(
    range(len(correlation.columns)),
    correlation.columns
)

plt.title("Correlation Between Rodri Performance Metrics")
plt.tight_layout()
plt.show()


# Phase 26 — Six-Dimension Performance Profile


In [ ]:
profile = pd.DataFrame({
    "Dimension": [
        "Ball Circulation",
        "Progression",
        "Defensive Contribution",
        "Pressing",
        "Off-Ball Availability",
        "Physical Output"
    ],

    "Main Metric": [
        "Pass Completion (%)",
        "Completed Line Breaks",
        "Possession Regains",
        "Direct + Indirect Pressing",
        "Total Offers",
        "Distance Covered (km)"
    ],

    "Value": [
        round(pass_accuracy, 1),
        int(match_df["line_breaks_completed"].sum()),
        int(match_df["possession_regains"].sum()),
        int(
            match_df["pressing_direct"].sum()
            + match_df["pressing_indirect"].sum()
        ),
        int(match_df["total_offers"].sum()),
        round(match_df["total_distance_m"].sum() / 1000, 2)
    ]
})

profile


# Phase 27 — Main Findings

## 1. Ball Circulation

Rodri attempted a very high volume of passes while maintaining excellent pass completion.

This suggests that he was a central part of Spain's possession structure.

## 2. Progression

His line-breaking statistics show that his contribution was not limited to safe sideways circulation.

A high number of successful line-breaking actions indicates that Rodri repeatedly helped Spain move possession beyond opposition pressure.

## 3. Defensive Impact

Possession regains, tackles, interceptions, clearances, and blocks capture contributions that goals and assists completely ignore.

## 4. Pressing

Direct and indirect pressing actions show how Rodri influenced opposition possession even when he was not the player making the final recovery.

## 5. Off-Ball Intelligence

Rodri's large number of passing offers shows that his contribution often started before receiving the ball.

His movement helped create passing options and maintain possession structure.

## 6. Physical Output

The distance data shows a very high workload across the tournament, supporting his role as a midfielder who constantly connected different phases of play.


# Phase 28 — Methodological Limitations

This project must be interpreted carefully.

### 1. The merged CSV contains multiple sub-datasets

The same football action may be present in different tables. We therefore use `source_dataset` to isolate each analysis.

### 2. Match-level and event-level pass totals differ

The match-level data and Opta event-level data use slightly different definitions and classifications.

Use:

- **match-level data** for official tournament summaries and match comparisons;
- **event-level pass data** for spatial pass visualizations.

### 3. No xG variable

The shot dataset does not contain expected goals, so the project does not invent xG-based analysis.

### 4. No direct assist variable in the merged dataset

The dataset directly supports the statement that Rodri scored zero goals, but a separate verified source would be required to independently prove zero assists.

### 5. Only Rodri is analyzed

This dataset explains Rodri's performance profile, but it cannot by itself prove that he was better than every other midfielder in the tournament.


# Phase 29 — Final Conclusion

This analysis demonstrates why goals and assists alone are insufficient for evaluating a central midfielder.

Rodri's World Cup 2026 performance was characterized by:

- high involvement in possession;
- excellent passing efficiency;
- successful line-breaking actions;
- defensive recoveries;
- pressing activity;
- off-ball availability;
- significant physical output.

His influence extended far beyond the final attacking action.

Rather than being primarily a scorer or assist provider, Rodri contributed to the mechanisms that allowed his team to control possession, progress the ball, recover it after losses, and maintain structure throughout matches.

## Final Answer to the Research Question

**A midfielder's true impact should be measured through a multidimensional profile combining possession, progression, defensive contribution, pressing, off-ball movement, and physical output rather than relying only on goals and assists.**
